In [ ]:
# %load_ext autoreload
%reload_ext autoreload
%autoreload 2
%load_ext line_profiler

import os
import time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from clearml import Task
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader
from mglyph_ml.dataset.utils import load_images_and_labels
from mglyph_ml.dataset.utils import show_datasets
from dataclasses import dataclass
import torch.functional as F


from mglyph_ml.dataset.glyph_dataset import GlyphDataset

In [ ]:
sracka = 190
task_name = "Stab 2"
task_tag = "stability-2"
dataset_path = Path("data/simple-star.mglyph")
seed = 67
max_steps = 10000 * 100 // 64
learning_rate = 0.0005
offline = True
batch_size = 64
data_loader_num_workers = 4
num_bins = 5
sample_count = 9999999999
hard_mining_warmup_steps = 200
hard_mining_fraction = 0.7
hard_mining_alpha = 0.6
hard_mining_ema = 0.9
hard_mining_eps = 1e-3

In [ ]:
Task.set_offline(offline)
task: Task = Task.init(project_name="mglyph-ml", task_name=task_name)
task.add_tags(task_tag)
load_dotenv()

task.connect(
    {
        "dataset_path": str(dataset_path),
        "seed": seed,
        "max_steps": max_steps,
        "batch_size": batch_size,
        "data_loader_num_workers": data_loader_num_workers,
        "sample_count": sample_count,
        "learning_rate": learning_rate,
        "num_bins": num_bins,
        "hard_mining_warmup_steps": hard_mining_warmup_steps,
        "hard_mining_fraction": hard_mining_fraction,
        "hard_mining_alpha": hard_mining_alpha,
        "hard_mining_ema": hard_mining_ema,
        "hard_mining_eps": hard_mining_eps,
    }
)

In [ ]:
loaded_split_0 = load_images_and_labels(
    dataset_path=dataset_path,
    split="0",
    desired_size=(64, 64),
    shuffle=True,
    seed=seed,
)

train_limit = min(sample_count, len(loaded_split_0.images))
images_train = loaded_split_0.images[:train_limit]
labels_train = loaded_split_0.labels[:train_limit]
images_overall = images_train
labels_overall = labels_train

In [ ]:
affine = A.Affine(
    rotate=(-3, 3),
    translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
    fit_output=False,
    keep_ratio=True,
    border_mode=cv2.BORDER_CONSTANT,
    fill=255,
    p=1.0,
)
normalize = A.Normalize(mean=0.0, std=1.0, max_pixel_value=255.0)
to_tensor = ToTensorV2()
pipeline = A.Compose([affine, normalize, to_tensor], seed=seed)
normalize_pipeline = A.Compose([normalize, to_tensor])


def affine_and_normalize(image: np.ndarray) -> torch.Tensor:
    return pipeline(image=image)["image"]


def just_normalize(image: np.ndarray) -> torch.Tensor:
    return normalize_pipeline(image=image)["image"]


dataset_train = GlyphDataset(
    name="Training",
    images=images_train,
    labels=labels_train,
    transform=just_normalize,
)
dataset_overall = GlyphDataset(
    name="Overall",
    images=images_overall,
    labels=labels_overall,
    transform=just_normalize,
)

print(f"train dataset size: {len(dataset_train)}")

In [ ]:
show_datasets(dataset_train, dataset_overall)

In [ ]:
from mglyph_ml.nn.glyph_regressor_binned import BinnedGlyphRegressor

device = os.environ["MGML_DEVICE"]
print(f"Training device: {device}")
model = BinnedGlyphRegressor(num_divisions=num_bins)

print(model.bin_centers_x)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# TODO: check if this is correct
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=(1 / 10) ** (1 / max_steps))

generator = torch.Generator().manual_seed(seed)

data_loader_train = DataLoader(
    dataset_train,
    batch_size=batch_size,
    num_workers=data_loader_num_workers,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True,
    generator=generator,
)

# data_loader_overall = DataLoader(
#     dataset_overall,
#     batch_size=batch_size,
#     num_workers=data_loader_num_workers,
#     pin_memory=True,
#     shuffle=False,
# )

In [ ]:
# benchmark

from time import perf_counter

for i in range(20):
    run_batches = 0
    run_samples = 0
    start = perf_counter()

    for images, values in data_loader_train:
        run_batches += 1
        run_samples += len(values)  # robust if last batch is smaller

    elapsed = perf_counter() - start
    sec_per_100 = elapsed / run_samples * 100
    print(f"Run {i+1}: {sec_per_100:.4f} sec per 100 samples")

In [ ]:
from collections import deque

from mglyph_ml.nn.util import plot_actual_vs_predicted_labels

model.to(device)

step = 0
running_loss_train = 0.0
running_error_train = 0.0
num_batches_train = 0

train_iterator = iter(data_loader_train)

while step < max_steps:
    model.train()

    try:
        inputs, labels = next(train_iterator)
    except StopIteration:
        train_iterator = iter(data_loader_train)
        inputs, labels = next(train_iterator)

    inputs = inputs.to(device, non_blocking=True).float()
    labels = labels.to(device, non_blocking=True).float().view(-1)

    optimizer.zero_grad()
    logits: torch.Tensor = model(inputs)
    preds = model.logits_to_labels(logits)

    loss = criterion(preds, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()

    error = torch.mean(torch.abs(preds - labels)).item()

    running_loss_train += loss.item()
    running_error_train += error
    num_batches_train += 1
    step += 1

    if step % 100 == 0 or step == max_steps:
        print("=" * 80)
        print(f"Step {step}: loss={loss.item()}, lr={scheduler.get_last_lr()[0]:.6f}")
        # task.logger.report_scalar(title="Loss", series="Train (window)", value=avg_train_loss, iteration=step)
        # task.logger.report_scalar(title="MAE (x-space)", series="Train (window)", value=avg_train_error, iteration=step)
        # print("=" * 80)
        # print(f"Step {step}")
        # print("-" * 80)
        # print(f"  Train Loss (last {len(recent_train_losses)}): {avg_train_loss:.6f}  |  Train MAE (x, last {len(recent_train_errors)}): {avg_train_error:.4f}  |  ")

    # if step % plot_every_n_steps == 0 or step == max_steps:
    #     fig = plot_actual_vs_predicted_labels(
    #         model=model,
    #         dataset=dataset_overall,
    #         device=device,
    #         sample_count=min(1000, len(dataset_overall)),
    #         seed=seed,
    #         title=f"Actual vs Predicted (step {step})",
    #     )
    #     training_progress_figures.append((step, fig))

# Overall evaluation kept disabled below for optional future use
    # model.eval()
    # running_loss_overall = 0.0
    # running_error_overall = 0.0
    # num_batches_overall = 0

    # with torch.no_grad():
    #     for inputs, labels in data_loader_overall:
    #         inputs = inputs.to(device)
    #         labels = labels.to(device)
    #         labels = labels.float().view(-1)

    #         logits = model(inputs)
    #         target_bins = model.labels_to_bins(labels)
    #         preds = model.logits_to_labels(logits, strategy="expectation")
    #         ce_loss = criterion_ce(logits, target_bins)
    #         mse_loss = criterion_mse(preds, labels)
    #         loss = ce_weight * ce_loss + mse_weight * mse_loss
    #         error = torch.mean(torch.abs(preds - labels)).item()

    #         running_loss_overall += loss.item()
    #         running_error_overall += error
    #         num_batches_overall += 1

    # avg_overall_loss = running_loss_overall / num_batches_overall
    # avg_overall_error = running_error_overall / num_batches_overall
    # avg_overall_error_x = avg_overall_error * 100.0

    # task.logger.report_scalar(title="Loss", series="Overall", value=avg_overall_loss, iteration=step)
    # task.logger.report_scalar(title="MAE (x-space)", series="Overall", value=avg_overall_error_x, iteration=step)

In [ ]:
import random as _py_random

print(f"Inference device: {device}")

model.eval()

n_samples = min(5000, len(dataset_train))
sample_indices = _py_random.sample(range(len(dataset_train)), n_samples)

x_true = []
x_pred = []

with torch.inference_mode():
    for idx in sample_indices:
        img_tensor, label = dataset_train[idx]
        img_tensor = img_tensor.unsqueeze(0).float().to(device, non_blocking=True)
        label_value = float(label)

        logits = model(img_tensor)
        pred_value = float(model.logits_to_labels(logits).squeeze(0).item())

        x_true.append(label_value)
        x_pred.append(pred_value)

x_true = np.array(x_true)
x_pred = np.array(x_pred)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x_true, x_pred, alpha=0.5, s=12)

min_x = float(min(np.min(x_true), np.min(x_pred)))
max_x = float(max(np.max(x_true), np.max(x_pred)))
ax.plot([min_x, max_x], [min_x, max_x], linestyle="--", linewidth=1.2, color="black", label="Ideal (y=x)")

ax.set_xlabel("Ground truth x")
ax.set_ylabel("Predicted x")
ax.set_title(f"Ground Truth vs Predicted x (random {n_samples} training samples)")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

In [ ]:
import random as _py_random

print(f"Inference device: {device}")

model.eval()

n_samples = min(1000, len(dataset_train))
sample_indices = _py_random.sample(range(len(dataset_train)), n_samples)

x_vals = []
losses_per_sample = []

with torch.inference_mode():
    for idx in sample_indices:
        img_tensor, label = dataset_train[idx]
        img_tensor = img_tensor.unsqueeze(0).float().to(device, non_blocking=True)
        label_value = float(label)

        logits = model(img_tensor)
        pred = model.logits_to_labels(logits).squeeze(0)

        label_tensor = torch.tensor([label_value], dtype=torch.float32, device=device)
        pred_tensor = pred.view(1)
        loss = criterion(pred_tensor, label_tensor).item()

        x_vals.append(label_value)
        losses_per_sample.append(loss)

x_vals = np.array(x_vals)
losses_per_sample = np.array(losses_per_sample)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_vals, losses_per_sample, alpha=0.5, s=12)
ax.set_xlabel("Actual label (raw x)")
ax.set_ylabel("Loss (training objective)")
ax.set_title(f"Loss vs. x (random {n_samples} training samples)")
plt.tight_layout()
plt.show()

In [ ]:
task.flush(wait_for_uploads=True)
task.close()